# Group Work AS 2026: ESG, Stock Returns and Machine Learning

**Course:** EAIF: AI for Finance

This notebook is the starting point for the group assignment. It loads the data, explores it, works through **one** forecasting model end to end and shows **one** backtest pattern. Everything after Section 4 is your work.

## Dataset

Simulated by the course for this assignment (licence CC BY 4.0, EAIF course). Five data files and a README, loaded from the course GitHub repository; the data dictionary handed out with the assignment describes every column.

> ### Simulated data with real problems
> There are no real firms behind these rows. The generator planted a small number of data problems of the kind real financial data has: things that are not what they seem, values that are known too early or too late, samples that are not what you think they are. **Finding and treating them is task (a) and is graded.** The generator also planted one genuine relationship between ESG scores and returns; whether you find it, and only it, is task (c).
>
> Your memo must still contain a section on what a real-data version of your analysis would need beyond what you did here. That section is graded (see the rubric below).

## How to use this notebook

1. Make a copy (File, then "Save a copy in Drive").
2. Run the given cells in order and read the comments.
3. Add your own cells for the tasks in Section 5. Keep the given cells so the grader can follow the pipeline.
4. Submit the completed notebook together with the memo and the slides.


# Assignment brief

**DEADLINE: <to be set by the instructor>**

**Group size:** 3 to 4 students. Assign roles at the first meeting and name them in the memo: a **data lead** (pipeline, audit), a **modelling lead** (models, cross-validation, tuning), an **analysis lead** (premium, backtest, figures, tables) and a **writing lead** (memo, slides). Everyone must be able to explain every cell.

**Deliverables**

| Deliverable | Format | Length |
|---|---|---|
| Notebook | `.ipynb`, runs top to bottom in Colab without errors | as needed |
| Memo | PDF, addressed to an investment committee | 3 pages plus appendix |
| Slides | PDF, for a 10-minute presentation | at most 8 slides |

**Rubric (100 points)**

| Criterion | Points | What the grader looks for |
|---|---|---|
| Pipeline and data audit (task a) | 20 | 5 for a leakage-free formation panel; 15 for the audit table, graded against the list of planted problems |
| Model comparison (task b) | 20 | linear baseline vs tree ensemble, selection on cross-validation, test reported once, next to a naive baseline |
| Where does ESG pay (task c) | 15 | 8 for finding the cell where the premium exists and only that cell; 4 for honest uncertainty; 3 for the point-in-time vs restated comparison |
| Backtest with costs and an ESG floor (task d) | 15 | formed after publication, held twelve months, costs charged, the cost of the floor shown, bootstrap of the difference to the benchmark |
| Memo and slides (task e) | 15 | clear question, clear answer, figures with labelled axes and units, no overclaiming; 10 minutes |
| What would change with real data | 15 | 5 for the list (what remains after your audit), 10 for which of your conclusions would survive and why |

Tasks (a) to (d) are graded from the notebook and the memo together: the memo states the result, the notebook must show how it was produced. A result in the memo that the notebook does not reproduce earns no points.

**Given / Your work**

| | Sections | Status |
|---|---|---|
| Given | 1 Setup and loading, 2 Exploration, 3 One worked model, 4 One backtest pattern | run and read; do not delete |
| Your work | 5 Tasks (a) to (e) | new cells below Section 5, plus the memo and the slides |


# 1. Setup and data loading (given)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")


In [ ]:
# Five data files, simulated by the course. Licence: CC BY 4.0 (EAIF course); cite them in the memo.
DATA_URL = os.environ.get("ESG2026_DATA", "https://raw.githubusercontent.com/umatter/EDFB/main/data/esg2026/")

panel    = pd.read_csv(DATA_URL + "firm_year_panel.csv")       # firm x fiscal year: fundamentals, point-in-time ESG scores
restated = pd.read_csv(DATA_URL + "esg_scores_restated.csv")   # firm x fiscal year: ESG scores as downloaded today
returns  = pd.read_csv(DATA_URL + "stock_returns_monthly.csv") # firm x month: total return, percent
factors  = pd.read_csv(DATA_URL + "factors_monthly.csv")       # month: risk-free rate, market, size, industry factors, percent
universe = pd.read_csv(DATA_URL + "universe_current.csv")      # firms in the universe at the end of the last year

for name, df in [("panel", panel), ("restated", restated), ("returns", returns), ("factors", factors), ("universe", universe)]:
    print(f"{name:<9} {df.shape[0]:>8,} rows x {df.shape[1]:>2} columns")

# Join keys: panel and restated on (Company_ID, Year); returns and factors on Month; universe on Company_ID.
# Publication_Date says when the fiscal-year row became public. Nothing in that row is known before that date.
print("\nPublication dates for fiscal year 2020 (first rows):")
print(panel.loc[panel["Year"] == 2020, ["Company_ID", "Year", "Publication_Date"]].head(3).to_string(index=False))
print("Latest publication date in the panel:", pd.to_datetime(panel["Publication_Date"]).dt.strftime("%m-%d").max(),
      "(so a portfolio formed on 1 July of the following year can use the whole fiscal year)")
panel.head()


# 2. Exploration (given)

Look before you model. Three things to check in every panel: what is missing and where, how sizes are distributed, and how many firms there are in each year. Nothing below is cleaned. Write down every oddity you notice; each one is a line in your audit table (task a).


In [ ]:
print("Missing values per column (panel):")
print(panel.isna().sum()[lambda s: s > 0])
print("\nSummary statistics (panel):")
display(panel.describe().round(2))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].hist(np.log10(panel["Revenue"]), bins=60, color="steelblue")
axes[0, 0].set_title("log10 revenue, all firm-years"); axes[0, 0].set_xlabel("log10(USD million)"); axes[0, 0].set_ylabel("Rows")

by_year = panel.groupby("Year")["Company_ID"].nunique()
axes[0, 1].bar(by_year.index, by_year.values, color="grey")
axes[0, 1].set_title("Firms with a fiscal-year row"); axes[0, 1].set_xlabel("Fiscal year"); axes[0, 1].set_ylabel("Firms")

esg_trend = panel.groupby(["Year", "Region"])["ESG_Score"].mean().unstack()
esg_trend.plot(ax=axes[1, 0], marker="o"); axes[1, 0].set_title("Average ESG score by region and year")
axes[1, 0].set_xlabel("Fiscal year"); axes[1, 0].set_ylabel("Score (0 to 100)"); axes[1, 0].set_ylim(0, 100)

axes[1, 1].hist(returns["Return_Pct"].clip(-40, 40), bins=80, color="purple", alpha=0.7)
axes[1, 1].set_title("Monthly total returns (clipped at ±40 for display)"); axes[1, 1].set_xlabel("Percent per month"); axes[1, 1].set_ylabel("Firm-months")
plt.tight_layout(); plt.show()

print("Firms per industry and region (panel):")
print(panel.groupby("Industry")["Company_ID"].nunique().to_string())
print(panel.groupby("Region")["Company_ID"].nunique().to_string())


**Before you go on, answer in your audit table:** what is the second cluster in the revenue histogram, and which firms are in it? Why does the number of firms change from year to year, and which file would hide that? Which columns are missing, for which firms, and is that random? Are there rows in the returns file that should not be there?


# 3. One worked model: forecasting the next twelve months' return (given)

This section shows the pattern you must follow in all your own models.

**The decision-time rule.** At the moment the decision is made, which columns are already known? We form portfolios on **1 July** of each year. By then every firm has published its previous fiscal year (the latest `Publication_Date` is at the end of May). So at formation in July of year $t+1$:

- every **feature** comes from the fiscal-year row of year $t$ (or earlier), plus returns up to June of $t+1$;
- the **target** is the total return from July of $t+1$ to June of $t+2$, compounded from the monthly file;
- the **target** needs twelve holding months to be well defined, so the model is fitted on firms that
  have them; firms that leave mid-year stay in the panel for the backtest in Section 4,
- the **split** is by formation year: train on formation years up to 2021, test on 2022 to 2024. A random split would put a firm's neighbouring years on both sides and let the model recognise firms instead of forecasting.

Every column you add as a feature in task (a) has to pass this rule. Ask of each one: *is it in a file that existed on 1 July, with the value it had then?*

`Year` is not a feature. Industry and region are one-hot encoded so that a linear model can use them (task b). The feature list below is deliberately short; extending it is your work.


In [ ]:
FORMATION_MONTH = 7
SCORES = ["ESG_Score", "Environmental_Score", "Social_Score", "Governance_Score"]

def holding_returns(fm, factors, formation_year):
    # 12-month total and excess return from July of formation_year to June of the next year
    start, end = f"{formation_year}-07", f"{formation_year + 1}-06"
    rf = factors.set_index("Month")["Rf_Pct"]
    w = fm[(fm["Month"] >= start) & (fm["Month"] <= end)].copy()
    w["gross"] = 1 + w["Return_Pct"] / 100
    w["gross_ex"] = 1 + (w["Return_Pct"] - w["Month"].map(rf)) / 100
    g = w.groupby("Company_ID")
    return pd.DataFrame({"Hold_Return_Pct": 100 * (g["gross"].prod() - 1),
                         "Hold_Excess_Pct": 100 * (g["gross_ex"].prod() - 1), "Months": g.size()}).reset_index()

def formation_panel(fy, fm, factors, features_from_panel):
    # one row per firm and formation year; features from fiscal year t, target = next 12-month return
    fy = fy.sort_values(["Company_ID", "Year"]).copy()
    fy["Growth_Lag1"] = fy.groupby("Company_ID")["Growth_Rate"].shift(1)
    fy["Formation_Year"] = fy["Year"] + 1
    published = pd.to_datetime(fy["Publication_Date"]) < pd.to_datetime(fy["Formation_Year"].astype(str) + f"-{FORMATION_MONTH:02d}-01")
    fy = fy[published]
    fy["log_Revenue"] = np.log(fy["Revenue"]); fy["log_Market_Cap"] = np.log(fy["Market_Cap"])
    years = sorted(fy["Formation_Year"].unique())
    past = pd.concat([holding_returns(fm, factors, y - 1).assign(Formation_Year=y) for y in years])
    past = past.rename(columns={"Hold_Return_Pct": "Past12_Return_Pct"})[["Company_ID", "Formation_Year", "Past12_Return_Pct"]]
    tgt = pd.concat([holding_returns(fm, factors, y).assign(Formation_Year=y) for y in years])
    # Keep every firm with at least one holding month, including the ones that leave during the year.
    tgt = tgt.rename(columns={"Months": "Hold_Months"})[
        ["Company_ID", "Formation_Year", "Hold_Return_Pct", "Hold_Excess_Pct", "Hold_Months"]]
    out = fy.merge(past, on=["Company_ID", "Formation_Year"], how="left").merge(tgt, on=["Company_ID", "Formation_Year"], how="inner")
    dummies = pd.get_dummies(out[["Industry", "Region"]], drop_first=True, dtype=int)
    out = pd.concat([out, dummies], axis=1)
    features = features_from_panel + ["Growth_Lag1", "log_Revenue", "log_Market_Cap", "Past12_Return_Pct"] + list(dummies.columns)
    return out.dropna(subset=features + ["Hold_Return_Pct"]).reset_index(drop=True), features

ml_data, feature_cols = formation_panel(panel, returns, factors, ["Growth_Rate", "Profit_Margin"] + SCORES)

# A firm needs a full twelve holding months to be a well-defined forecasting target, so we fit on
# those. Firms that leave during the holding year stay in ml_data: Section 4 holds them until their
# last month, which is what a real backtest does. Dropping them here instead would quietly remove the
# firms that did worst before they disappeared.
model_data = ml_data[ml_data["Hold_Months"] == 12]
print(f"Firm-formation-years: {len(ml_data):,} ({len(model_data):,} with a full twelve holding months), "
      f"formation years {ml_data['Formation_Year'].min()}-{ml_data['Formation_Year'].max()}")
print(f"Features ({len(feature_cols)}): {feature_cols}")

train_mask = model_data["Formation_Year"] <= 2021
test_mask = model_data["Formation_Year"] >= 2022
X, y, groups = model_data[feature_cols], model_data["Hold_Return_Pct"], model_data["Company_ID"]
X_train, y_train, g_train = X[train_mask], y[train_mask], groups[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

baseline_pred = np.full(len(y_test), y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
print(f"\nNaive baseline (training mean = {y_train.mean():.2f}%): test RMSE = {baseline_rmse:.3f}")

cv = GroupKFold(n_splits=5)
candidates = {
    "RF depth 3": RandomForestRegressor(n_estimators=300, max_depth=3, min_samples_leaf=20, random_state=42, n_jobs=-1),
    "RF depth 6": RandomForestRegressor(n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=42, n_jobs=-1),
}
cv_results = {}
for name, model in candidates.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, groups=g_train, scoring="neg_root_mean_squared_error")
    cv_results[name] = (-scores.mean(), scores.std())
    print(f"{name:<12} CV RMSE = {-scores.mean():.3f} (sd across folds {scores.std():.3f})")
best_name = min(cv_results, key=lambda k: cv_results[k][0])
rf_model = candidates[best_name].fit(X_train, y_train)
pred_test = rf_model.predict(X_test)
summary = pd.DataFrame({
    "RMSE": [np.sqrt(mean_squared_error(y_train, rf_model.predict(X_train))), cv_results[best_name][0], np.sqrt(mean_squared_error(y_test, pred_test)), baseline_rmse],
    "R2": [r2_score(y_train, rf_model.predict(X_train)), np.nan, r2_score(y_test, pred_test), r2_score(y_test, baseline_pred)],
}, index=["Train (in-sample)", "CV (grouped, train years)", "Test (2022-2024, once)", "Naive baseline on test"])
print(f"\nSelected on CV: {best_name}\n"); print(summary.round(3))

importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
importance.head(10)[::-1].plot(kind="barh", ax=axes[0]); axes[0].set_title("Top 10 feature importances"); axes[0].set_xlabel("Importance")
axes[1].scatter(y_test, pred_test, alpha=0.3, s=10); lims = [y_test.min(), y_test.max()]
axes[1].plot(lims, lims, "r--", label="perfect forecast"); axes[1].set_xlabel("Actual 12-month return (%)"); axes[1].set_ylabel("Forecast (%)")
axes[1].set_title(f"Test formation years 2022-2024 (R2 = {r2_score(y_test, pred_test):.3f})"); axes[1].legend()
plt.tight_layout(); plt.show()


**Reading the result.** Returns are hard to forecast: in this run the selected forest's test RMSE is 37.307 percentage points against 36.931 for the naive mean, and the test R2 is -0.480 (your numbers may differ slightly). The forest does not beat the naive mean here, and that is the honest result, not a bug. Two things are worth understanding rather than working around. First, a negative R2 means the model also does worse than simply predicting the test years' own average; that happens whenever the average return in the test period differs from the training period, which is what a market downturn inside the test years does. Second, a model that does much better than this on the test years has usually seen something it should not have. Note what carries the importance and ask, for every feature you add in task (a), whether it is known on 1 July.


# 4. One backtest pattern: formed after publication, held twelve months (given)

A portfolio rule must use only information available when the portfolio is formed. The pattern below ranks all firms on 1 July of each year on a score from the fiscal-year row published before that date, holds the top decile equally weighted until the next 30 June, and records the monthly portfolio return. Firms that leave the panel during the year contribute their returns until their last month; the remaining names are re-weighted equally. A transaction cost of 20 basis points per unit of one-way turnover is charged at each rebalancing (sell and buy, so 40 basis points on the replaced share).

Over the monthly series we report the annualised return, the annualised volatility, and the **Sharpe ratio** (mean monthly excess return over the risk-free rate divided by its standard deviation, times the square root of twelve). The equal-weight portfolio of all firms in the panel is the benchmark every rule must beat.

The given example ranks on the **ESG score**. Task (d) ranks on your **forecast** from task (b), adds an ESG floor, and asks what the floor costs.


In [ ]:
def backtest(picks, fm, factors, cost_bp=20):
    # picks: dict formation_year -> list of Company_ID held from July of that year to June of the next
    rf = factors.set_index("Month")["Rf_Pct"]
    months, rets, turns, prev = [], [], [], set()
    for y in sorted(picks):
        held = set(picks[y])
        turn = 1.0 if not prev else len(held - prev) / max(len(held), 1)
        turns.append(turn)
        window = fm[fm["Company_ID"].isin(held) & (fm["Month"] >= f"{y}-07") & (fm["Month"] <= f"{y + 1}-06")]
        monthly = window.groupby("Month")["Return_Pct"].mean()
        if len(monthly):
            monthly.iloc[0] -= 2 * cost_bp / 100 * turn
        months += list(monthly.index); rets += list(monthly.values); prev = held
    r = pd.Series(rets, index=months)
    ex = r - r.index.map(rf)
    return {"Annualised return (%)": 100 * ((1 + r / 100).prod() ** (12 / len(r)) - 1),
            "Annualised volatility (%)": r.std() * np.sqrt(12),
            "Sharpe ratio": ex.mean() / ex.std() * np.sqrt(12),
            "Mean one-way turnover": np.mean(turns), "Months": len(r)}

formation_years = sorted(y for y in ml_data["Formation_Year"].unique() if y >= 2015)
esg_picks, all_picks = {}, {}
for yr in formation_years:
    at_t = ml_data[ml_data["Formation_Year"] == yr]
    n_top = max(int(0.1 * len(at_t)), 1)
    esg_picks[yr] = at_t.nlargest(n_top, "ESG_Score")["Company_ID"].tolist()
    all_picks[yr] = at_t["Company_ID"].tolist()

results = pd.DataFrame({"Top decile by ESG score": backtest(esg_picks, returns, factors),
                        "All firms, equal weight": backtest(all_picks, returns, factors, cost_bp=0)}).T
print(f"Formation years {formation_years[0]}-{formation_years[-1]}, holding July to June:")
print(results.round(3))


**Reading the table.** In this run the ESG-ranked decile earned a Sharpe ratio of 0.175 against 0.212 for the equal-weight panel. Before you read anything into that, remember Section 2: high-ESG firms sit in particular industries and regions, so the decile is partly an industry and region bet; and a Sharpe ratio over 126 monthly observations has a wide confidence band. Task (d) asks you to bootstrap the difference. Note also that ranking the whole world by ESG score is a blunt rule: if a relation between ESG and returns exists only among certain firms, or only in certain years, a single global ranking pooled over the whole sample will bury it. That is what task (c) asks you to look for. And note which firms the backtest was run on: the panel, including the firms that left. Run it on the current universe instead and see what changes.


# 5. Your tasks

Add your cells below this section. Each task ends with a one-line deliverable that must appear in the memo.

**(a) Pipeline and data audit.** Build your own formation panel from the point-in-time file with your own feature choices; add at least three features that are not in the worked model and justify each with the decision-time rule (is it in a file that existed on 1 July, with the value it had then?). Before that, audit the five data files: what is duplicated, what is in the wrong unit, what is missing and for whom, what is known too early, which file is a biased sample. Treat each problem and say how. The test set must be separated by formation year; `GroupKFold` by company is for the cross-validation inside the training years, as in Section 3. *Deliverable: a one-page audit table (issue, how detected, how treated, effect on one headline number), and a table of your features with the date from which each is known.*

**(b) Linear baseline vs tree ensembles.** Fit a linear regression (unit 01) on standardised features and at least one of random forest or gradient boosting (unit 03) for the twelve-month return. Tune the ensembles on cross-validation inside the training years. Report train, CV and test performance side by side, next to the naive mean baseline, for every model. Select the model on CV, and report the test result once. "The ensemble does not pay off out of sample" is a legitimate result and is graded as such. *Deliverable: one comparison table and one sentence on whether the extra complexity of the ensemble pays off out of sample.*

**(c) Where does ESG pay?** Estimate the relation between the previous fiscal year's Environmental score (standardised within year) and the next twelve months' excess return, by region and by period (before and from 2021), with industry-by-year controls. Report point estimates with bootstrap or clustered standard errors and say where the relation is distinguishable from zero and where it is not. Repeat the estimate with the restated scores file and explain the difference. *Deliverable: one table (region x period, point-in-time and restated) and one sentence on where an investor could have earned an ESG premium, and where not.*

**(d) Backtest with costs and an ESG floor.** Rank on the forecast of your selected model from (b), trained for each formation year on earlier formation years only; hold the top decile from July to June; charge 20 basis points per unit of one-way turnover. You may either re-tune the model's settings inside each formation year's own training window, or tune once on the training years and keep that setting for every year. Both are accepted; say which you did and why. Tuning once is a simplification, because the setting was then chosen with information from years the early forecasts should not have seen, even though no return value enters those fits. Add an ESG floor: the pool is restricted to firms whose Environmental score is at or above the 50th, then the 80th percentile of that year. Report annualised return, volatility, Sharpe ratio and turnover for no floor, the two floors, and the equal-weight benchmark, all on the full panel including the firms that left. Bootstrap the Sharpe difference between your best rule and the benchmark (resample months). *Deliverable: one table and one figure of the cumulative return series, and one sentence on what the floor costs and whether the difference to the benchmark is distinguishable from noise.*

**(e) Memo and slides.** Write a 3-page memo to an investment committee: the question, the data (state that it is simulated), the audit, the pipeline, the results of (b) to (d), and a section titled "What would change with real data" that lists what remains to be solved after an audit like yours (scores that are proprietary and disagree across raters, survivorship that is harder to see, costs that are not a constant, point-in-time availability) and says which of your conclusions would survive and which would not, and why. Prepare at most 8 slides for a 10-minute presentation. *Deliverable: the memo PDF and the slides PDF, with the CC BY 4.0 attribution of the dataset.*

**Rules.** Every metric appears next to the baseline it must beat. Every figure has labelled axes with units. Every feature is dated. Cite the dataset with its licence.
